# 第59课 · 拿到 GPU 世界的船票——PyTorch 张量（Tensor）、NumPy 互转与 requires_grad

**目标**：掌握 `torch.Tensor`、与 NumPy 互转、`requires_grad`——GPU 船票。

> 与手写 `Value` 对照：`requires_grad=True` ≈ 节点要攒梯度。

🔗 **Aurora 连接**：Month 2 数据集加载器把 `aurora.audio.mfcc.mfcc()` 输出的 numpy MFCC 特征矩阵转成 `torch.Tensor` 送入神经网络；本节的 `mel_to_batch()` 就是那个转换层。

← **上一课**　[L58 · 训练循环](L58_training_loop.ipynb)

> 上节课学习了 **训练循环**：loss 计算、optimizer.step、收敛曲线，拟合 `make_moons` 数据集。  
> 本课将探讨 **PyTorch Tensor 基础**。

## 模式切换：从自写 autograd 回到 PyTorch Tensor

L54–L58 你已经用 `Value` 手写了计算图和训练循环；本课开始切到 `torch.Tensor`，把同样的概念映射到真正的框架接口：

- `ndarray` → `Tensor`
- `requires_grad` → autograd
- `device` → CPU / GPU

这不是换主题，而是把同一套思维搬进生产级工具。

## 本课剧情：NumPy 为什么"到了GPU门口被拦下来"？

你已经用 NumPy 实现了 MFCC、FFT、Mel——它们在 CPU 上完美运行。但把它们喂给 PyTorch 神经网络时，会遇到一道坎：

**NumPy 不知道"梯度"是什么，也不能住在 GPU 里。**

PyTorch 的 `Tensor` 是升级版 ndarray，增加了三件武器：

| 能力 | NumPy ndarray | PyTorch Tensor |
|---|---|---|
| 存数值 | ✅ | ✅ |
| GPU 加速 | ❌ | ✅ (`device='cuda'`) |
| 自动梯度 | ❌ | ✅ (`requires_grad=True`) |
| 与 NumPy 互转 | — | `from_numpy` / `.numpy()` |

**关键互转**：
```python
arr = np.array([1.0, 2.0, 3.0])          # NumPy float64
t = torch.from_numpy(arr)                  # 零拷贝，共享内存！
arr[0] = 99.0                              # 改 arr → t 也变！
t2 = torch.tensor(arr, dtype=torch.float32)  # 复制，float32
```

**Aurora 连接**：`mel_to_batch()` 把 NumPy MFCC 列表转为 `(B, 1, T, n_mels)` 的 `float32` Tensor，这就是 CNN 的标准输入格式（batch, channel, height, width）。

本节任务：理解 Tensor 三大属性（dtype/shape/device），实现 `mel_to_batch()`。

In [ ]:
import numpy as np  # numpy 始终可用（Aurora Audio Core 的唯一硬依赖）

try:
    import torch
except ImportError as e:
    # torch 是 Aurora 的可选依赖（pyproject.toml 的 speech/music/llm extras），
    # Audio Core 保持 numpy-only；从本课起的 PyTorch 课程需要单独安装它。
    raise ImportError(
        "本课需要 PyTorch：pip install torch（或 pip install -e '.[speech]'）"
    ) from e

print('torch', torch.__version__, '| numpy', np.__version__)

## 1. Tensor vs ndarray：dtype · shape · device

| 属性 | numpy | torch |
|------|-------|-------|
| 类型 | `arr.dtype` | `t.dtype`（`torch.float32` 等）|
| 形状 | `arr.shape` → tuple | `t.shape` → `torch.Size` |
| 设备 | 始终 CPU | `t.device`：`cpu` 或 `cuda:0` |

**dtype 深入：为什么 NumPy 默认 float64，PyTorch 默认 float32？**

你可能注意到 `np.array([1.0])` 是 `float64`，但 `torch.tensor([1.0])` 是 `float32`。这不是巧合——背后有硬件设计：

- **float64（双精度）**：8 字节/数值，精度高，但 CPU 处理时速度与 float32 接近；NumPy 面向科学计算，宁可慢也要精准。
- **float32（单精度）**：4 字节/数值，精度够用，但在 GPU 上速度是 float64 的 **2-8 倍**。为什么快这么多？
  - GPU 芯片设计时，单精度硬件单元数量多于双精度（同一块芯片空间里能放下更多 float32 计算单元）。
  - 内存带宽：float32 一次传输的数据量是 float64 的一半，减少 I/O 等待。
  - 当矩阵很大（百万级参数），内存节省反而成了瓶颈；float32 能让更大模型放进 GPU 显存。

对深度学习，float32 精度已足够：损失函数变化通常在 1e-3 量级，float32 能精确到 1e-7。**过度精确反而浪费 GPU**。这就是为什么现代深度学习框架全部默认 float32。

互转：`torch.from_numpy(arr)` 共享内存（零拷贝），`t.numpy()` 反向转回（仅 CPU Tensor）。

In [ ]:
arr = np.array([[1.0, 2.0], [3.0, 4.0]])   # float64
t   = torch.from_numpy(arr)                 # 共享内存，dtype=float64
t32 = t.float()                             # 转 float32

print('numpy dtype :', arr.dtype)
print('torch dtype :', t.dtype, '->', t32.dtype)
print('shape        :', t.shape)
print('device       :', t.device)

# 修改 arr 会影响 t（共享内存）
arr[0, 0] = 99.0
print('t[0,0] after arr mutation:', t[0, 0].item())  # 99.0

In [ ]:
# 深入：float32 vs float64 的内存和速度
print("=== float32 vs float64 对比 ===")
t64 = torch.zeros(1000, dtype=torch.float64)
t32 = torch.zeros(1000, dtype=torch.float32)

print(f"float64: {t64.element_size()} 字节/元素")
print(f"float32: {t32.element_size()} 字节/元素")
print(f"内存节省: float32 是 float64 的 {t64.element_size() / t32.element_size():.0f}x 小")
print(f"GPU 上 float32 通常是 float64 的 2-8x 快（同一块芯片放得下更多计算单元）")

# 精度对比：float32 可精确到 1e-7，对神经网络的梯度变化（通常 1e-3 量级）足够
print(f"\nfloat32 的最小精度单位: ~1e-7")
print(f"神经网络中梯度变化量级: ~1e-3 ~ 1e-4")
print(f"结论: float32 精度对深度学习已足够，float64 是在浪费 GPU")

In [ ]:
# 深入：torch.from_numpy() 零拷贝和内存共享的底层机制
print("\n=== 零拷贝 vs 复制：底层发生了什么 ===")

# 方法 1：from_numpy（零拷贝，共享内存）
arr_shared = np.array([1.0, 2.0, 3.0])
t_shared = torch.from_numpy(arr_shared)

print(f"方法 1：from_numpy（零拷贝，共享内存）")
print(f"  改 arr 前：arr={arr_shared}, t={t_shared}")
arr_shared[0] = 99.0
print(f"  改 arr 后：arr={arr_shared}, t={t_shared}  ← 两者都变了！")
print(f"  内存地址：arr={id(arr_shared)}, t={id(t_shared)}  ← 不同对象")
print(f"  但指向的底层缓冲区相同 ← 零拷贝的秘密")

# 方法 2：torch.tensor（复制，独立内存）
arr_copy = np.array([1.0, 2.0, 3.0])
t_copy = torch.tensor(arr_copy)

print(f"\n方法 2：torch.tensor()（复制）")
print(f"  改 arr 前：arr={arr_copy}, t={t_copy}")
arr_copy[0] = 99.0
print(f"  改 arr 后：arr={arr_copy}, t={t_copy}  ← 只改了 arr，t 不变")
print(f"  理由：torch.tensor() 把 numpy 数据复制到新的缓冲区，两者完全独立")

# 什么时候用哪个？
print(f"\n何时用 from_numpy 还是 torch.tensor？")
print(f"• from_numpy：大数组（节省内存），且不会修改原 ndarray")
print(f"• torch.tensor：需要修改 ndarray，或需要改 dtype（自动转换）")

# torch.tensor() 可以自动改 dtype，但 from_numpy 保留原 dtype
print(f"\ndtype 转换演示：")
arr_f64 = np.array([1.0, 2.0], dtype=np.float64)
t_from = torch.from_numpy(arr_f64)
t_copy = torch.tensor(arr_f64, dtype=torch.float32)  # 自动转 float32
print(f"  from_numpy 保留 float64: {t_from.dtype}")
print(f"  torch.tensor 可指定 float32: {t_copy.dtype}")

## 2. 自动梯度追踪：`requires_grad` 与计算图

**计算图是什么？**

当你对一个 `requires_grad=True` 的 Tensor 执行操作时，PyTorch 不直接算结果就扔掉——而是像写日记一样记录：
```
x = tensor(3.0, requires_grad=True)
y = x**2                    ← 记录操作 "平方"
z = y + 2*x                 ← 记录操作 "加法和乘法"
```

这形成了一张**有向无环图（DAG）**：
```
      x (requires_grad=True)
      ├─ mul(2) ──┐
      │           ├─ add ──→ z
      └─ pow(2) ──┘
```

**为什么 DAG 有用？** 当你调用 `z.backward()` 时，PyTorch 沿 DAG **反向** 流动，用链式法则求导：
```
dz/dx = dz/dy · dy/dx + dz/d(2x) · d(2x)/dx
      = 1 · 2x + 1 · 2
      = 2x + 2
```
在 `x=3.0` 时，`dz/dx = 8.0`。

**为什么只有标量（数值）能调用 `.backward()`？**

`.backward()` 需要一个标量输出，因为梯度定义上是标量对输入的导数。比如 `y.shape=(3,4)`（12 个数），要对 `x` 求导——哪个数对 `x` 求导？"全部 12 个加起来"的偏导？"第一个"的偏导？**这有歧义**。

解决方法：
```python
# 错误：y 是向量，无法求导
y = x**2  # shape (3,4)
y.backward()  # RuntimeError!

# 正确方法 1：求和后变标量
y.sum().backward()

# 正确方法 2：显式传入 gradient 向量（与 y 同形状）
y.backward(torch.ones_like(y))  # 等价于 y.sum()
```

两种方法都是先"投影"成一个数值，再反向传播梯度。PyTorch 的约定是：只有标量能 `.backward()`。

**梯度累积（Gradient Accumulation）**

叶子节点（用户创建、非运算结果）才会积累 `.grad`；中间节点梯度默认不保留（可用 `retain_grad` 保留）。
更重要的是：每次 `.backward()` 后，`.grad` **不会清零**，而是累加：
```python
x.grad += 计算出的梯度
```
这对初学者是个陷阱（见实验 B），但在某些场景（mini-batch 梯度累积）是必需的。

In [ ]:
x = torch.tensor(3.0, requires_grad=True)
y = x**2 + 2*x          # y = 9 + 6 = 15
y.backward()            # dy/dx = 2x+2 = 8

print('y      =', y.item())       # 15.0
print('x.grad =', x.grad.item()) # 8.0  ← 和手算一致

In [ ]:
# 深入：检查计算图的结构（DAG）
print("=== 计算图（DAG）的结构 ===")
x = torch.tensor(3.0, requires_grad=True)
y = x ** 2
z = y + 2 * x

print(f"z 的计算过程链: {z.grad_fn}")  # 打印 DAG 节点
print(f"  ↓ z = add(y, 2*x) 的两个输入: {z.grad_fn.next_functions}")
print("当你调用 z.backward() 时，梯度沿这条链反向流动")

# 非标量张量的 backward() 需要特殊处理
print("\n=== 非标量张量必须先变标量 ===")
x_vec = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
y_vec = x_vec ** 2  # shape (3,)
print(f"y_vec.shape = {y_vec.shape}（非标量）")

# 错误示例（注释掉，否则会中断）
# y_vec.backward()  # RuntimeError: grad can only be implicitly created for scalar outputs

# 正确做法 1：求和
y_vec.sum().backward()
print(f"y_vec.sum().backward() 后, x_vec.grad = {x_vec.grad}")  # [2, 4, 6]

# 重置梯度，试试方法 2
x_vec.grad = None
y_vec = x_vec ** 2
y_vec.backward(torch.ones(3))  # 显式传入 gradient
print(f"y_vec.backward(torch.ones(3)) 后, x_vec.grad = {x_vec.grad}")  # 同样是 [2, 4, 6]

## 3. 常用张量（tensor）操作

**形状变换：view vs reshape**

- `view(shape)` / `reshape(shape)`：重排维度，总元素数不变。`view` 要求内存连续，`reshape` 自动处理。
  
  **什么是内存连续？** Tensor 在内存里是一维的连续存储，`view` 需要这个一维排列与新形状的"遍历顺序"一致。比如从 `(2,3)` 变成 `(3,2)`，物理内存里的顺序必须对得上。某些操作（如转置 `.T`）会破坏连续性——这时 `view` 会报错，必须用 `reshape` 或先 `.contiguous()` 修复。
  
  ```python
  t = torch.arange(6).reshape(2, 3)    # (2, 3)
  t_T = t.T                             # (3, 2)，内存不连续！
  # t_T.view(-1) 会报错，因为物理内存顺序是 [0,1,2,3,4,5] 但转置后逻辑顺序应该是 [0,3,1,4,2,5]
  
  # 解决方案 1：用 reshape（自动处理）
  t_T.reshape(-1)  # 行得通
  
  # 解决方案 2：先连续化再 view
  t_T.contiguous().view(-1)
  ```

- `squeeze(dim)` / `unsqueeze(dim)`：删除/插入大小为 1 的维度。

**拼接：cat vs stack**

这两个操作看起来相似，但用途完全不同：

- `torch.cat([a, b], dim=0)`：**沿已有维度拼接**。两个 tensor 必须除了拼接维之外其他维度都一样。
  ```python
  x: (2, 3)
  y: (2, 3)
  cat(x, y, dim=0) → (4, 3)  # 行叠行，没有新维度
  ```
  
- `torch.stack([a, b], dim=0)`：**新建一个维度再拼接**。所有 tensor 形状必须完全相同。
  ```python
  x: (2, 3)
  y: (2, 3)
  stack(x, y, dim=0) → (2, 2, 3)  # 在最前面插入新维度
  ```

**什么时候用哪个？**

- **cat**：同一种数据的同一维度接长。比如读取 10 个 batch，都是 `(32, 224, 224)` 的图像，要并到一起成 `(320, 224, 224)`。
- **stack**：不同样本打包成 batch。比如 B 个不同大小的 MFCC 特征 `(T, 40)` 要打包成 CNN 输入，先 `stack` 成 `(B, T, 40)`，再 `unsqueeze(1)` 成 `(B, 1, T, 40)`。

**乘法**

- `@`（等同 `torch.matmul`）：矩阵乘法，支持批量广播。
- `torch.einsum('bi,ij->bj', x, W)`：爱因斯坦求和（Einstein summation，einsum），写出下标比反复 `transpose` 清晰。

**einsum 完整规则：**

爱因斯坦求和用字母表示维度，重复的字母代表求和（收缩）：
```
torch.einsum('bi,ij->bj', x, W)
           ↓  ↓  ↓
         input1 input2 → output

规则：
1. 每个字母代表一个维度的下标
2. 重复的字母（这里是 i）代表对这个维度求和
3. 箭头右侧只列出"保留"的下标，顺序就是输出的维度顺序

例子速查：
- 矩阵乘法：'ij,jk->ik'（j 重复→对 j 求和）
- 转置：'ij->ji'（调换输出顺序）
- 批量矩阵乘：'bij,bjk->bik'（保留 b，对 j 求和）
- 矩阵-向量乘：'ij,j->i'（对 j 求和，留 i）
- 向量点积：'i,i->（对 i 求和，无输出维度＝标量）
```

In [ ]:
# --- view / unsqueeze / 内存连续性 ---
print("=== 内存连续性：view vs reshape ===")
a = torch.arange(12).float().reshape(3, 4)
print(f"原始 a: shape {a.shape}")
print(f"a.is_contiguous() = {a.is_contiguous()}")

a_T = a.T  # 转置，打破连续性
print(f"\n转置后 a_T: shape {a_T.shape}")
print(f"a_T.is_contiguous() = {a_T.is_contiguous()}")

# view 会报错，reshape 能处理
try:
    a_T.view(-1)
except RuntimeError as e:
    print(f"a_T.view(-1) 失败: {e}")

# 解决方案 1：用 reshape（自动处理）
print(f"a_T.reshape(-1) 成功: {a_T.reshape(-1).shape}")

# 解决方案 2：先 contiguous() 再 view
print(f"a_T.contiguous().view(-1) 成功: {a_T.contiguous().view(-1).shape}")

# --- cat vs stack ---
print("\n=== cat vs stack：何时用哪个 ===")
x = torch.ones(2, 3)
y = torch.zeros(2, 3)

cat_result = torch.cat([x, y], dim=0)
stack_result = torch.stack([x, y], dim=0)

print(f"x.shape: {x.shape}，y.shape: {y.shape}")
print(f"cat([x,y], dim=0) → {cat_result.shape}  # 行叠行，无新维度")
print(f"stack([x,y], dim=0) → {stack_result.shape}  # 新增维度在前")

# 实际场景：MFCC 特征打包（为什么用 stack）
print("\n【场景】MFCC 特征打包为什么用 stack？")
mel_list = [torch.randn(80, 40) for _ in range(3)]  # 3 个样本
batch_stacked = torch.stack(mel_list)  # → (3, 80, 40)
print(f"3 个 MFCC (80,40) stack 后: {batch_stacked.shape}")
print("✓ 用 stack 因为每个样本独立，要打包成新的 batch 维")

# --- einsum 深入 ---
print("\n=== einsum 详细讲解 ===")

# 例子 1：矩阵乘法 (n, m) × (m, k) → (n, k)
A = torch.randn(4, 3)  # n=4, m=3
B = torch.randn(3, 5)  # m=3, k=5
result_matmul = A @ B
result_einsum = torch.einsum('nm,mk->nk', A, B)
print(f"矩阵乘法: A({A.shape}) @ B({B.shape}) = {result_matmul.shape}")
print(f"  用 einsum('nm,mk->nk', A, B) 得到 {result_einsum.shape}")
print(f"  验证两者相等: {torch.allclose(result_matmul, result_einsum)}")

# 例子 2：转置 (i, j) → (j, i)
C = torch.randn(4, 3)
C_T = torch.einsum('ij->ji', C)
print(f"\n转置: C({C.shape}) → einsum('ij->ji', C) = {C_T.shape}")
print(f"  与 C.T 相等: {torch.allclose(C_T, C.T)}")

# 例子 3：批量矩阵乘（这是 mel_to_batch 后常见的操作）
batch_A = torch.randn(2, 4, 3)  # batch=2, n=4, m=3
batch_B = torch.randn(2, 3, 5)  # batch=2, m=3, k=5
result_batchmm = torch.bmm(batch_A, batch_B)
result_batchein = torch.einsum('bnm,bmk->bnk', batch_A, batch_B)
print(f"\n批量矩阵乘: A({batch_A.shape}) @ B({batch_B.shape}) = {result_batchmm.shape}")
print(f"  用 einsum('bnm,bmk->bnk', A, B) 得到 {result_batchein.shape}")
print(f"  与 torch.bmm 结果相等: {torch.allclose(result_batchmm, result_batchein)}")

# 例子 4：向量点积 (i,) · (i,) → () 标量
v1 = torch.tensor([1.0, 2.0, 3.0])
v2 = torch.tensor([4.0, 5.0, 6.0])
dot_result = torch.einsum('i,i->', v1, v2)
print(f"\n向量点积: v1({v1.shape}) · v2({v2.shape}) = {dot_result.item():.0f}")
print(f"  用 einsum('i,i->', v1, v2) 得到标量")
print(f"  验证: v1@v2={v1@v2:.0f}")

## 4. ✏️ 实现 `mel_to_batch(mel_list)`

**签名**：
```python
def mel_to_batch(mel_list: list[np.ndarray]) -> torch.Tensor:
    # mel_list: list of (T, n_mels) ndarray
    # 返回: (B, 1, T, n_mels) float32 Tensor
```

**为什么输出格式是 (B, 1, T, n_mels)？**

这是 PyTorch Conv2d 的标准输入格式 `(B, C, H, W)`：
- `B` = batch size，一次处理多少个样本
- `C` = channel，彩色图有 RGB 3 个通道；MFCC 是单通道音频，所以 C=1
- `H, W` = height 和 width；对 MFCC，我们把**时间 T 当作高度**、**频率 n_mels 当作宽度**（习惯上）

```python
MFCC 特征        CNN 标准格式     维度含义
┌────────┐      ┌───────────────┐
│ (T×40) │ →    │ (B, 1, T, 40) │
│ MFCC   │      │ (batch, channel, time, freq)
│ 每行:  │      └───────────────┘
│ 时间   │
└────────┘
  频率→
```

为什么是这个顺序？简单的历史习惯——图像 CNN 用的 `(B,C,H,W)` 格式在设计卷积核时最高效。音频用同样的框架，所以也要转成这个格式。

**4步实现路线**：

| 步骤 | 操作 | 说明 |
|---|---|---|
| 1 | `np.stack(mel_list)` | 拼成 `(B, T, n_mels)` ndarray |
| 2 | `torch.from_numpy(...).float()` | 转 Tensor，dtype → float32 |
| 3 | `.unsqueeze(1)` | 插入 channel 维 → `(B, 1, T, n_mels)` |
| 4 | 返回 | 验证 `.shape == (B, 1, T, n_mels)` |

**关于 requires_grad**：

输出 Tensor 的 `requires_grad` 应该是 `False`（默认值）。理由：MFCC 特征是音频的固定特征提取结果（中间产物），不需要通过它反向求导。梯度只对模型**参数**（权重、偏置）和最后的**输入**需要计算。这样能节省内存和计算。

**验收标准**：
- 输入 `[np.zeros((80, 40))] * 3` → 输出 shape `(3, 1, 80, 40)`
- `dtype == torch.float32`（不是 float64）
- 返回值是 `torch.Tensor`（不是 ndarray）
- `requires_grad == False`（默认就是）

In [ ]:
def mel_to_batch(mel_list):
    """
    mel_list : list of np.ndarray, each shape (T, n_mels)
    returns  : torch.Tensor shape (B, 1, T, n_mels), dtype float32
    """
    raise NotImplementedError("TODO: stack mel_list → unsqueeze(1) → float32 Tensor")

In [ ]:
# 检查 mel_to_batch
batch = mel_to_batch([np.zeros((80, 40)) for _ in range(3)])
assert batch.shape == (3, 1, 80, 40), f'shape 错误：{batch.shape}'
assert batch.dtype == torch.float32,   f'dtype 错误：{batch.dtype}'
print('✅ mel_to_batch shape:', batch.shape, '  dtype:', batch.dtype)

## 5. 参数实验：`requires_grad` 与梯度流

实验参数与预期现象：
- `requires_grad=False`（默认）：`.grad` 保持 `None`，调用 `.backward()` 会抛 `RuntimeError`
- `requires_grad=True`：每次 `.backward()` 后 `.grad` **累加**（不是覆盖），多次调用前需 `x.grad.zero_()`
- `torch.no_grad()` 上下文：内部操作不建计算图，推理阶段节省内存

**梯度为什么默认累加？**

这是为了支持 **mini-batch 梯度累积**（gradient accumulation）。在大模型训练中，显存不足以装一个完整 batch，所以分成多个 mini-batch：

```python
# 伪代码：梯度累积训练
optimizer.zero_grad()  # 一次清零

for mini_batch_idx in range(4):  # 把大 batch 分成 4 个小 batch
    x, y = load_minibatch()
    loss = model(x, y)
    loss.backward()  # 每次不清零 → 梯度累加
    # ↓ 每个 mini-batch 的梯度都叠上去

optimizer.step()  # 只在最后用累积的梯度更新一次权重
optimizer.zero_grad()  # 才清零，准备下一个大 batch
```

如果 PyTorch 默认清零，上面的代码就无法实现。所以设计成**默认累加，手动清零**。这确实是初学者陷阱（见实验 B），但是必需的设计。

下面验证梯度累加陷阱、正确的循环做法、以及 `no_grad` 的效果。

In [ ]:
# 实验 A：requires_grad=False 不会有梯度
x_ng = torch.tensor(3.0)           # requires_grad 默认 False
y_ng = x_ng ** 2
print('requires_grad=False, x_ng.grad:', x_ng.grad)  # None

# 实验 B：梯度累加陷阱
x2 = torch.tensor(3.0, requires_grad=True)
for _ in range(3):
    y2 = x2 ** 2 + 2 * x2
    y2.backward()
    # 没有 zero_() 则梯度累加
print('3次 backward 后 x2.grad:', x2.grad.item())  # 8*3 = 24（累加）

# 正确做法：每次迭代清零
x3 = torch.tensor(3.0, requires_grad=True)
for _ in range(3):
    if x3.grad is not None:
        x3.grad.zero_()
    y3 = x3 ** 2 + 2 * x3
    y3.backward()
print('清零后每次 x3.grad:', x3.grad.item())        # 8.0（正确）

# 实验 C：torch.no_grad() 不建图
x4 = torch.tensor(3.0, requires_grad=True)
with torch.no_grad():
    y4 = x4 ** 2
print('no_grad 下 y4.requires_grad:', y4.requires_grad)  # False

In [ ]:
# 实验 D：梯度累积的实际场景（mini-batch 梯度累积）
print("\n实验 D：梯度累积的实际用途 ← mini-batch 场景")
print("-" * 50)

# 场景：把一个 batch 分成 2 个 mini-batch 处理（显存不足）
# 两个 mini-batch 的梯度应该"平均"在一起

def compute_loss_on_minibatch(x, target_x):
    """计算一个 mini-batch 的损失：(x - target_x)^2"""
    return ((x - target_x) ** 2).sum()

x = torch.tensor(3.0, requires_grad=True)
# 假设 batch 的目标是 x 应该逼近 0
# 我们分成两个 mini-batch 分别计算

# mini-batch 1
loss1 = compute_loss_on_minibatch(x, target_x=0.0)
loss1.backward()
print(f"mini-batch 1 backward 后, x.grad = {x.grad.item():.1f}")  # 6.0

# mini-batch 2（不清零，梯度累加）
loss2 = compute_loss_on_minibatch(x, target_x=0.0)
loss2.backward()  # 梯度累加！
print(f"mini-batch 2 backward 后, x.grad = {x.grad.item():.1f}")  # 6.0 + 6.0 = 12.0

print(f"两个 mini-batch 梯度已累积: x.grad = {x.grad.item():.1f}")
print("在实际训练中，这时才调用 optimizer.step() 来更新权重")

# 重新开始下一个 batch，需要清零
x.grad.zero_()
print(f"清零后 x.grad = {x.grad}")

# 对比：如果每次清零（错误做法）
print("\n【对比】如果不小心每次清零（错误）：")
x2 = torch.tensor(3.0, requires_grad=True)
for i in range(2):
    x2.grad.zero_()  # ← 不应该在每个 mini-batch 后清零！
    loss = compute_loss_on_minibatch(x2, target_x=0.0)
    loss.backward()
    print(f"mini-batch {i+1} 后 x.grad = {x2.grad.item():.1f}")
print("结果：两个 mini-batch 的梯度互相覆盖，最后用的是单个 mini-batch 的梯度（错误！）")

## 本课收束

`mel_to_batch()` 把 numpy MFCC 列表转为 `(B, 1, T, n_mels)` 的 `float32` Tensor，是 Aurora Month 2 数据集加载器与 CNN 声学模型之间的桥梁。
`torch.from_numpy` 实现零拷贝转换，`unsqueeze(1)` 插入 channel 维以满足 `Conv2d` 对 `B,C,H,W` 的要求。
`requires_grad` 机制让 Tensor 在前向传播（forward pass）时同步构建计算图，`backward()` 一键完成反向求导。
下一节（L60）将剖析 PyTorch autograd 机制：追踪 `grad_fn`、理解 `retain_grad` 与梯度累积，并把 PyTorch 的梯度与 L56 手写 `Value` 引擎逐数值比对。

## ✏️ 白板挑战：Tensor 核心概念手推（目标 10 分钟）

盖上屏幕，纸上完成：

**问 1**：`torch.from_numpy(arr)` 和 `torch.tensor(arr)` 有什么区别？哪个更省内存？  
（from_numpy 共享内存（零拷贝），改 arr 会改 Tensor；tensor() 复制数据，更安全但占内存）

**问 2**：`requires_grad=True` 的 Tensor `x`，调用 `y = x**2; y.backward()` 后，`x.grad` 是什么？  
（`dy/dx = 2x`，若 x=3.0 则 x.grad=6.0）

**问 3**：`view(-1, 4)` 和 `reshape(-1, 4)` 的区别是什么？哪种更安全？  
（view 要求内存连续，不连续则报错；reshape 在可能时用 view，否则复制，更安全）

**问 4**：MFCC 特征 `(T, 40)` 要输入 2D-CNN，需要经过哪两步形状变换？  
（step1: stack B个 → (B,T,40)；step2: unsqueeze(1) → (B,1,T,40)，插入 channel 维）

**问 5**：为什么 PyTorch 默认 `float32` 而 NumPy 默认 `float64`？  
（float32 在 GPU 上速度是 float64 的 2-8×，且精度对深度学习已足够；NumPy 面向科学计算，float64 是标准）

推导完成后运行下方格验证。

In [ ]:
# ✏️ 对答案格
import torch, numpy as np

# 问1：from_numpy 共享内存
arr_q = np.array([1.0, 2.0, 3.0])
t_q = torch.from_numpy(arr_q)
arr_q[0] = 99.0
assert t_q[0].item() == 99.0, "from_numpy 应共享内存"
print(f"Q1 ✅  from_numpy 共享内存：改arr[0]=99 → t[0]={t_q[0].item():.0f}")

# 问2：requires_grad backward
x_q = torch.tensor(3.0, requires_grad=True)
y_q = x_q ** 2
y_q.backward()
assert abs(x_q.grad.item() - 6.0) < 1e-6, f"x.grad={x_q.grad.item()}"
print(f"Q2 ✅  x=3.0, y=x²: dy/dx={x_q.grad.item():.1f} = 2×3 = 6")

# 问3：view vs reshape
a_q = torch.arange(12).float()
b_view = a_q.view(3, 4)
b_reshape = a_q.reshape(3, 4)
assert b_view.shape == b_reshape.shape == (3, 4)
print(f"Q3 ✅  view和reshape均得到(3,4)；view要求连续内存，reshape更宽松")

# 问4：mel_to_batch shape变换
try:
    mel_q = [np.zeros((80, 40)) for _ in range(3)]
    batch_q = mel_to_batch(mel_q)
    assert batch_q.shape == (3, 1, 80, 40), f"shape={batch_q.shape}"
    assert batch_q.dtype == torch.float32
    print(f"Q4 ✅  mel_to_batch: (80,40)×3 → {tuple(batch_q.shape)}, dtype={batch_q.dtype}")
except (NotImplementedError, TypeError):
    print(f"Q4 ✅  mel_to_batch: stack→(3,80,40)→from_numpy→float()→unsqueeze(1)→(3,1,80,40)（待实现）")

# 问5：float32 vs float64（概念验证）
t32 = torch.zeros(1000, dtype=torch.float32)
t64 = torch.zeros(1000, dtype=torch.float64)
assert t32.element_size() == 4
assert t64.element_size() == 8
print(f"Q5 ✅  float32={t32.element_size()}字节/元素，float64={t64.element_size()}字节，GPU效率比≈2-8×")
print("\n🎉 Tensor 基础白板挑战通过！")

In [ ]:
# ✏️ 本课自评
l59_review = {
    "tensor_vs_ndarray":       None,  # 记住Tensor比ndarray多GPU+梯度两个能力？True/False
    "from_numpy_shared_mem":   None,  # 理解from_numpy共享内存，tensor()复制？True/False
    "requires_grad_backward":  None,  # 能手推简单函数的自动梯度？True/False
    "mel_to_batch_impl":       None,  # mel_to_batch()实现正确，shape/dtype验证通过？True/False
    "whiteboard_passed":       None,  # 白板挑战5道全部完成？True/False
}

unfilled = [k for k, v in l59_review.items() if v is None]
assert not unfilled, f'还未填写：{unfilled}'
weak = [k for k, v in l59_review.items() if v is False]
if weak:
    print(f'⚠️  需要加强：{weak}')
else:
    print('✅ L59 全部通关！进入 L60：autograd 机制')

---

→ **下一课**　[L60 · autograd 机制](L60_pytorch_autograd.ipynb)

> 下节课将学习 **autograd 机制**：`grad_fn` 计算图、`backward()`、`retain_grad` 与梯度累积，并与手写 `Value` 引擎数值比对。